In [9]:
import os
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sksurv.ensemble import RandomSurvivalForest
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import GridSearchCV

In [10]:
def c_index_scorer(estimator, X, y):
    risk_scores = estimator.predict(X)
    return concordance_index_censored(y['event'], y['time'], risk_scores)[0]

def prepare_survival_data(event, time):
    return np.array([(bool(e), t) for e, t in zip(event, time)],
                    dtype=[('event', bool), ('time', float)])

In [15]:
fold = 1

X_train = np.load(f'matrix/X_train_{fold}.npy')
all_time = np.load(f'matrix/all_time_{fold}.npy')
all_event = np.load(f'matrix/all_event_{fold}.npy')

X_test = np.load(f'matrix/X_test_{fold}.npy')
all_time_val = np.load(f'matrix/all_time_val_{fold}.npy')
all_event_val = np.load(f'matrix/all_event_val_{fold}.npy')

In [16]:
output_file = f"c_index_results_fold.csv"

In [23]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train, train_data)

train_risk_rsf = rsf.predict(X_train)
val_risk_rsf = rsf.predict(X_test)

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.9046, Val_C_Index Total: 0.7095


In [17]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [5, 10, 20],
}


results = []

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_fold = prepare_survival_data(all_event_val, all_time_val)

for train_idx, test_idx in kf.split(X_train, all_event):
    X_train_fold, X_test_fold = X_train[train_idx], X_train[test_idx]
    train_data_fold = prepare_survival_data(all_event[train_idx], all_time[train_idx])
    val_data_fold = prepare_survival_data(all_event[test_idx], all_time[test_idx])

    rsf = RandomSurvivalForest(random_state=42)
    grid = GridSearchCV(rsf, param_grid, cv=10, scoring=c_index_scorer, n_jobs=-1)
    grid.fit(X_train_fold, train_data_fold)

    best_model = grid.best_estimator_
    train_risk_rsf = best_model.predict(X_train_fold)
    val_risk_rsf = best_model.predict(X_test_fold)
    test_risk_rsf = best_model.predict(X_test)
    
    
    c_index_train = concordance_index_censored(
        train_data_fold['event'], train_data_fold['time'], train_risk_rsf)[0]
    c_index_val = concordance_index_censored(
        val_data_fold['event'], val_data_fold['time'], val_risk_rsf)[0]
    c_index_test = concordance_index_censored(
        test_fold['event'], test_fold['time'], test_risk_rsf)[0]

    results.append({
        'Fold': fold,
        'n_estimators': grid.best_params_['n_estimators'],
        'max_depth': grid.best_params_['max_depth'],
        'min_samples_split': grid.best_params_['min_samples_split'],
        'C_Index_Train': c_index_train,
        'C_Index_Val': c_index_val,
        'C_index_Test': c_index_test
    })

df_results = pd.DataFrame(results)

if os.path.isfile(output_file):
    df_results.to_csv(output_file, mode='a', header=False, index=False)
else:
    df_results.to_csv(output_file, mode='w', header=True, index=False)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [5, 10, 20],
}

results = []
best_model_overall = None
best_test_c_index = 0.5

train_data_fold = prepare_survival_data(all_event, all_time)
test_fold = prepare_survival_data(all_event_val, all_time_val)

ksf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
rsf = RandomSurvivalForest(random_state=0)
grid = GridSearchCV(rsf, param_grid, cv=ksf, scoring=c_index_scorer, n_jobs=-1)
grid.fit(X_train, train_data_fold)

best_model = grid.best_estimator_
train_risk_rsf = best_model.predict(X_train)
test_risk_rsf = best_model.predict(X_test)

c_index_train = concordance_index_censored(
    train_data_fold['event'], train_data_fold['time'], train_risk_rsf)[0]
c_index_test = concordance_index_censored(
    test_fold['event'], test_fold['time'], test_risk_rsf)[0]

results.append({
    'Fold': fold,
    'n_estimators': grid.best_params_['n_estimators'],
    'max_depth': grid.best_params_['max_depth'],
    'min_samples_split': grid.best_params_['min_samples_split'],
    'C_Index_Train': c_index_train,
    'C_Index_Test': c_index_test
})

df_results = pd.DataFrame(results)

if os.path.isfile(output_file):
    df_results.to_csv(output_file, mode='a', header=False, index=False)
else:
    df_results.to_csv(output_file, mode='w', header=True, index=False)

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd


rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train[:, :6], prepare_survival_data(all_event, all_time))

groups = {
    "Cliniche/Demografiche": (0, 6),
    "Genomiche": (6, 46),
    "Trascrittomiche": (46, 2067),
    "Immagini": (2067, 2131),
}

feature_importance = rsf.feature_importances_

group_importance = {}
for group, (start, end) in groups.items():
    group_importance[group] = np.sum(feature_importance[start:end])

group_importance = dict(sorted(group_importance.items(), key=lambda x: x[1], reverse=True))

plt.figure(figsize=(10, 6))
plt.barh(list(group_importance.keys()), list(group_importance.values()), color='skyblue')
plt.xlabel("Feature Importance")
plt.title("Importanza aggregata delle feature per gruppo")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
df_importance = pd.DataFrame({
    "Group": list(group_importance.keys()),
    "Importance": list(group_importance.values())
})

# Salva i risultati in un file CSV
df_importance.to_csv("feature_group_importance.csv", index=False)

print("Importanza aggregata salvata in 'feature_group_importance.csv'")